In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import random
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import random

In [2]:
pip install kagglehub


Note: you may need to restart the kernel to use updated packages.


In [3]:

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 3



In [4]:
import os

base_path = "/Users/nicoleriepl/.cache/kagglehub/datasets/amerzishminha/forest-fire-smoke-and-non-fire-image-dataset/versions/3/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET"

# List what’s inside to confirm structure
for root, dirs, files in os.walk(base_path):
    print(root, dirs, files)
    break


/Users/nicoleriepl/.cache/kagglehub/datasets/amerzishminha/forest-fire-smoke-and-non-fire-image-dataset/versions/3/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET ['test', 'train'] []


In [5]:
# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

# Paths
base_dir = '/kaggle/input/forest-fire-smoke-and-non-fire-image-dataset/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')

# Data Generator
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

In [6]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input  # or whichever model you're using
import os

# ✅ Base directory (your local dataset path)
BASE_DIR = "/Users/nicoleriepl/.cache/kagglehub/datasets/amerzishminha/forest-fire-smoke-and-non-fire-image-dataset/versions/3/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET"

# ✅ Subdirectories
train_dir = os.path.join(BASE_DIR, "train")
test_dir = os.path.join(BASE_DIR, "test")

# ✅ Data Generators
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2  # use 20% of train for validation
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# ✅ Image size and batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# ✅ Train and Validation generators
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

# ✅ Test data generator
test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


Found 25919 images belonging to 3 classes.
Found 6479 images belonging to 3 classes.
Found 10500 images belonging to 3 classes.


In [7]:
# Model 2: Transfer Learning with EfficientNetB0
def create_transfer_learning_model():
    # Load pre-trained EfficientNetB0
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    # Freeze base model initially
    base_model.trainable = False
    
    # Add custom classification head
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ])
    
    return model, base_model


In [8]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Input shape
IMG_SHAPE = (224, 224, 3)

# Load the base MobileNetV2 model without the top layers
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SHAPE)
base_model.trainable = False  # Freeze base layers for initial training

# Add custom classification head
inputs = Input(shape=IMG_SHAPE)
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(3, activation='softmax')(x)

mobilenetv2_model = Model(inputs, outputs)

# Compile the model
mobilenetv2_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
# Summary of the model
mobilenetv2_model.summary()

# Train the model (initial training phase with frozen base)
history_mobilenet = mobilenetv2_model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    verbose=1
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10


/opt/anaconda3/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


810/810 ━━━━━━━━━━━━━━━━━━━━ 761s 922ms/step - accuracy: 0.8094 - loss: 0.4742 - val_accuracy: 0.9542 - val_loss: 0.1648
Epoch 2/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 683s 838ms/step - accuracy: 0.9557 - loss: 0.1359 - val_accuracy: 0.9571 - val_loss: 0.1513
Epoch 3/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 690s 848ms/step - accuracy: 0.9676 - loss: 0.1009 - val_accuracy: 0.9596 - val_loss: 0.1415
Epoch 4/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 630s 773ms/step - accuracy: 0.9716 - loss: 0.0925 - val_accuracy: 0.9622 - val_loss: 0.1337
Epoch 5/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 663s 815ms/step - accuracy: 0.9742 - loss: 0.0808 - val_accuracy: 0.9626 - val_loss: 0.1397
Epoch 6/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 652s 801ms/step - accuracy: 0.9755 - loss: 0.0719 - val_accuracy: 0.9639 - val_loss: 0.1276
Epoch 7/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 609s 748ms/step - accuracy: 0.9779 - loss: 0.0671 - val_accuracy: 0.9653 - val_loss: 0.1292
Epoch 8/10
810/810 ━━━━━━━━━━━━━━━━━━━━ 656s 806ms/step - accuracy: 0.9794 - loss: 0.06

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# =========================
# 1️⃣ PATHS & DATA GENERATORS
# =========================
BASE_DIR = "/Users/nicoleriepl/.cache/kagglehub/datasets/amerzishminha/forest-fire-smoke-and-non-fire-image-dataset/versions/3/FOREST_FIRE_SMOKE_AND_NON_FIRE_DATASET"
train_dir = os.path.join(BASE_DIR, "train")
test_dir  = os.path.join(BASE_DIR, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=True
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

# =========================
# 2️⃣ MODEL DEFINITION
# =========================
IMG_SHAPE = (224, 224, 3)

# Base model
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SHAPE)
base_model.trainable = False  # freeze base initially

# Custom head
inputs = Input(shape=IMG_SHAPE)
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(train_data.num_classes, activation='softmax')(x)

mobilenetv2_model = Model(inputs, outputs)

mobilenetv2_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

mobilenetv2_model.summary()

# =========================
# 3️⃣ CALLBACKS
# =========================
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "mobilenetv2_fire_best.keras",
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

# =========================
# 4️⃣ INITIAL TRAINING (BASE FROZEN)
# =========================
history_initial = mobilenetv2_model.fit(
    train_data,
    validation_data=val_data,
    epochs=50,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

# =========================
# 5️⃣ FINE-TUNING (UNFREEZE TOP LAYERS)
# =========================
base_model.trainable = True  # unfreeze base for fine-tuning

# Optional: freeze bottom layers, fine-tune top layers only
# for layer in base_model.layers[:-50]:
#     layer.trainable = False

mobilenetv2_model.compile(
    optimizer=Adam(learning_rate=1e-5),  # lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_finetune = mobilenetv2_model.fit(
    train_data,
    validation_data=val_data,
    epochs=20,
    callbacks=[early_stopping, checkpoint],
    verbose=1
)

# =========================
# 6️⃣ EVALUATE ON TEST DATA
# =========================
test_loss, test_acc = mobilenetv2_model.evaluate(test_data, verbose=1)
print(f"Test Accuracy: {test_acc*100:.2f}%")


Found 25919 images belonging to 3 classes.
Found 6479 images belonging to 3 classes.
Found 10500 images belonging to 3 classes.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,422,339 (9.24 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/50
810/810 ━━━━━━━━━━━━━━━━━━━━ 0s 597ms/step - accuracy: 0.8153 - loss: 0.4625
Epoch 1: val_accuracy improved from -inf to 0.95123, saving model to mobilenetv2_fire_best.keras
810/810 ━━━━━━━━━━━━━━━━━━━━ 629s 764ms/step - accuracy: 0.8154 - loss: 0.4622 - val_accuracy: 0.9512 - val_loss: 0.1740
Epoch 2/50
810/810 ━━━━━━━━━━━━━━━━━━━━ 0s 633ms/step - accuracy: 0.9567 - loss: 0.1328
Epoch 2: val_accuracy improved from 0.95123 to 0.95385, saving model to mobilenetv2_fire_best.keras
810/810 ━━━━━━━━━━━━━━━━━━━━ 645s 792ms/step - accuracy: 0.9567 - loss: 0.1328 - val_accuracy: 0.9539 - val_loss: 0.1655
Epoch 3/50
810/810 ━━━━━━━━━━━━━━━━━━━━ 0s 625ms/step - accuracy: 0.9676 - loss: 0.1051
Epoch 3: val_accuracy improved from 0.95385 to 0.95786, saving model to mobilenetv2_fire_best.keras
810/810 ━━━━━━━━━━━━━━━━━━━━ 646s 793ms/step - accuracy: 0.9676 - loss: 0.1051 - val_accuracy: 0.9579 - val_loss: 0.1555
Epoch 4/50
810/810 ━━━━━━━━━━━━━━━━━━━━ 0s 637ms/step - accuracy: 0.9717 - l

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# =========================
# 1️⃣ PLOT TRAINING HISTORY
# =========================
def plot_history(history, title="Model Training"):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']
    epochs = range(1, len(acc) + 1)

    plt.figure(figsize=(14,5))

    # Accuracy
    plt.subplot(1,2,1)
    plt.plot(epochs, acc, 'b-', label='Training Accuracy')
    plt.plot(epochs, val_acc, 'r-', label='Validation Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss
    plt.subplot(1,2,2)
    plt.plot(epochs, loss, 'b-', label='Training Loss')
    plt.plot(epochs, val_loss, 'r-', label='Validation Loss')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()

# Plot initial training
plot_history(history_mobilenet, title="Initial Training")

# Plot fine-tuning
plot_history(history_finetune, title="Fine-Tuning")

# =========================
# 2️⃣ EVALUATE ON TEST SET
# =========================
test_loss, test_acc = mobilenetv2_model.evaluate(test_data, verbose=1)
print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

# =========================
# 3️⃣ CONFUSION MATRIX
# =========================
# Get true labels and predicted labels
y_true = test_data.classes
y_pred_probs = mobilenetv2_model.predict(test_data, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# Class labels
class_labels = list(test_data.class_indices.keys())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
plt.figure(figsize=(8,6))
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title("Confusion Matrix")
plt.show()

# =========================
# 4️⃣ CLASSIFICATION REPORT
# =========================
report = classification_report(y_true, y_pred, target_names=class_labels)
print("Classification Report:\n")
print(report)
